# Chapitre 7 - Cours 2 : isolation faible, read committed et snapshot isolation

Ce notebook couvre le début de la section *Weak Isolation Levels* : *Read Committed*
(*No dirty reads*, *No dirty writes*, *Implementing read committed*) et *Snapshot
Isolation and Repeatable Read* (*Implementing snapshot isolation*,
*Visibility rules for observing a consistent snapshot*, *Indexes and snapshot
isolation*, *Repeatable read and naming confusion*).
Les conflits entre écrivains (mises à jour perdues, write skew, fantômes) sont l'objet du
cours 3.

**Plan.**

1. Pourquoi des niveaux d'isolation « faibles » ? Les bogues de concurrence sont rares.
2. Read committed : pas de lecture sale.
3. Read committed : pas d'écriture sale.
4. Implémenter read committed : verrous d'écriture et deux versions.
5. Lecture biaisée (*read skew*) et instantanés.
6. MVCC : plusieurs versions de chaque ligne.
7. Les règles de visibilité.
8. Index, B-arbres immuables et ramasse-miettes.
9. Snapshot isolation, repeatable read : la confusion des noms.

Tous les scénarios sont rejoués sur le mini-moteur MVCC de
[`ddia/transactions.py`](../../ddia/transactions.py), qui implémente les niveaux
`read_uncommitted`, `read_committed`, `snapshot`, `serializable_ssi`,
`serializable_2pl` (et `none`, sans aucune protection).

In [1]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import transactions as tr
from ddia.transactions import R, W, S, run, plot_history, Engine, LEVEL_LABELS
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)